# Guidelines for Basic Robot Model Generation

Basic robot MJCF files in [mjcf_models](/d:/Leon/Python/RBS/robotblockset/mujoco/mjcf_models) should use **local names without the `BaseName` prefix**. This is how the provided models such as [panda.xml](/d:/Leon/Python/RBS/robotblockset/mujoco/mjcf_models/panda.xml), [ur10e.xml](/d:/Leon/Python/RBS/robotblockset/mujoco/mjcf_models/ur10e.xml), and [iiwa14.xml](/d:/Leon/Python/RBS/robotblockset/mujoco/mjcf_models/iiwa14.xml) are written.

Later, when generating a complete scene, the tutorial adds the robot prefix automatically, so names become `<BaseName>_...`. The MuJoCo wrappers in [robots_mujoco.py](/d:/Leon/Python/RBS/robotblockset/mujoco/robots_mujoco.py) and [robots_pymujoco.py](/d:/Leon/Python/RBS/robotblockset/mujoco/robots_pymujoco.py) then work with these prefixed names in the generated scene.

## 1. Important rule

- In the **basic robot MJCF model**, do not use the prefix `<BaseName>_`.
- In the **generated scene**, the prefix `<BaseName>_` is added automatically.
- The same rule applies to bodies, joints, sites, sensors, actuators, and also to all `default class` names used inside the robot model.

So the standalone robot model should contain names like:

- `base`, `link1`, `link7`
- `joint1`, `joint2`, `wrist_3_joint`
- `flange`, `TCP`, `FT`
- `pos`, `ori`, `v`, `w`, `force`, `torque`
- `robot`, `visual`, `collision`, `size4`, `joint1`

and not names like `Panda_joint1`, `Panda_TCP`, `Panda_pos`, or `Panda_robot` inside the basic model file.

## 2. Base robot body / base link

- The root body in the standalone model should be named with a local name, usually `base`.
- Internal links should also use local names such as `link0`, `link1`, `base_link`, `shoulder_link`, etc.
- Do not prefix these names inside the basic model.

Recommended:
```xml
<body name="base" pos="0 0 0">
    ...
</body>
```

After scene generation, this becomes something like `Panda_base` or `UR10e_base`.

## 3. Joint naming

- Use local joint names in the standalone model.
- Generic models can use `joint1`, `joint2`, ..., `jointN`.
- Robot-specific names such as `shoulder_pan_joint` or `wrist_3_joint` are also fine.
- Do not add `<BaseName>_` inside the basic model file.
- If special joint names are used, these names must also be known to the Python robot wrapper.
- This can be done by defining `self.joint_names` in the corresponding robot specification in [robot_spec.py](/d:/Leon/Python/RBS/robotblockset/robot_spec.py), or by passing `JointNames=...` when creating the robot object, as supported e.g. by [robot_mujoco](/d:/Leon/Python/RBS/robotblockset/mujoco/robots_mujoco.py).

Recommended:
```xml
<joint name="joint1" .../>
<joint name="joint2" .../>
<joint name="joint3" .../>
```

or, for robot-specific naming:
```xml
<joint name="shoulder_pan_joint" .../>
<joint name="elbow_joint" .../>
<joint name="wrist_3_joint" .../>
```

When the scene is generated, these names are prefixed automatically, for example `Panda_joint1` or `UR10e_wrist_3_joint`.

This point is important for robots such as UR10e, where joints are not named `joint1`, `joint2`, ... in the MJCF model. In that case, the same special names must be provided in the robot specification or explicitly through constructor arguments so that the wrapper can map scene joints correctly.

## 4. Actuator naming

- Actuator names in the standalone model should also be local, without `<BaseName>_`.
- The default actuator names are now sequential names such as `actuator1`, `actuator2`, ..., `actuatorN`.
- The actuator type is still determined by the MJCF element itself, for example `<position .../>` or `<general .../>`, not by the actuator name.
- If robot-specific actuator names are needed, they should still remain local and unprefixed inside the standalone model.

Recommended:
```xml
<position name="actuator1" joint="joint1" .../>
<position name="actuator2" joint="joint2" .../>
```

See [panda.xml](/d:/Leon/Python/RBS/robotblockset/mujoco/mjcf_models/panda.xml) for actuator definitions.

These become `<BaseName>_actuator1`, `<BaseName>_actuator2`, ... in the generated scene.

## 5. Sensor naming

Sensor names in the basic model should also be unprefixed.

Recommended local names:

- Joint position sensors: `pos_joint1`, ..., `pos_jointN`
- Joint velocity sensors: `vel_joint1`, ..., `vel_jointN`
- Cartesian position sensor: `pos`
- Cartesian orientation sensor: `ori`
- Cartesian linear velocity sensor: `v`
- Cartesian angular velocity sensor: `w`
- Force sensor: `force`
- Torque sensor: `torque`

Typical MJCF sensor block in the standalone model:
```xml
<sensor>
    <jointpos    name="pos_joint1" joint="joint1"/>
    <jointvel    name="vel_joint1" joint="joint1"/>
    <framepos    name="pos" objtype="site" objname="TCP"/>
    <framequat   name="ori" objtype="site" objname="TCP"/>
    <framelinvel name="v" objtype="site" objname="TCP"/>
    <frameangvel name="w" objtype="site" objname="TCP"/>
    <force       name="force" site="FT"/>
    <torque      name="torque" site="FT"/>
</sensor>
```

When the robot is inserted into a scene, these names become `<BaseName>_pos`, `<BaseName>_ori`, `<BaseName>_force`, etc.

## 6. Sites for flange and TCP

Two sites are especially important in the standalone robot model:

- `flange`: mounting interface of the bare robot arm
- `TCP`: actual tool center point after the mounted tool or gripper offset

Meaning:

- `flange` defines where a gripper or tool is mechanically attached.
- `TCP` defines the operational tool center point used for task-space motion and Cartesian sensing.
- The wrappers expect these sites as `<BaseName>_flange` and `<BaseName>_TCP` only after the scene generator has added the prefix.
- If both sites exist, the wrappers compute the transform from flange to TCP automatically.

Recommended pattern inside the last robot link or tool body:
```xml
<body name="link7" ...>
    ...
    <site name="flange" pos="0 0 0" quat="1 0 0 0" size="0.003" rgba="1 0 0 1"/>
    <site name="TCP"    pos="0 0 0.1034" quat="1 0 0 0" size="0.004" rgba="0 1 0 1"/>
</body>
```

Notes:

- If no tool is mounted, `TCP` can initially coincide with `flange`.
- If a gripper is mounted, keep `flange` at the robot mounting face and move `TCP` to the real tool center point.
- If you use a force/torque sensor site, a local name such as `FT` is also appropriate.
- Keep these sites rigidly attached to the last robot link or tool body.

## 7. Default classes

- `default class` names in the basic model must also be local and unprefixed.
- Use names such as `robot`, `visual`, `collision`, `finger`, `size4`, `joint1`, etc.
- Do not define classes such as `Panda_robot`, `Panda_visual`, or `UR10e_size4` in the standalone model.
- When generating the scene, these class names are also prefixed automatically.

Example:
```xml
<default class="robot">
    <joint damping="1" armature="0.1"/>
    <default class="visual">
        <geom group="2"/>
    </default>
</default>
```

## 8. Minimal naming checklist for standalone robot MJCF files

- Base/root body: local name such as `base`
- Links: local names such as `link1`, `link7`, `base_link`, `wrist_3_link`
- Joints: local names such as `joint1 ... jointN` or `shoulder_pan_joint ... wrist_3_joint`
- Actuators: local names such as `actuator1 ... actuatorN`
- Joint sensors: `pos_jointi`, `vel_jointi`
- Flange site: `flange`
- TCP site: `TCP`
- Optional FT site: `FT`
- Cartesian sensors: `pos`, `ori`, `v`, `w`, `force`, `torque`
- Default classes: local names such as `robot`, `visual`, `collision`, `size4`, `joint1`

In short: keep the standalone robot MJCF file self-contained and unprefixed. The `<BaseName>_` prefix is added later when the full scene is generated.

# Imports

In [2]:
import numpy as np
from robotblockset.tools import get_rbs_path, print_xml

from robotblockset.transformations import rot_x, rot_y, rot_z, map_pose

import mujoco 
from robotblockset.mujoco.tools_mjcf import print_body_tree, replace_in_mjcf_file

# Initialization

Define the folder where the MJCF models are

In [3]:
MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/"

Define objects to be added to scene

In [4]:
ROBOT = "panda_no_hand"
ROBOT_NAME = "panda"
GRIPPER = "panda_hand"
CAMERA = "realsense_d435i"
TOOL = "calibration_plate"
PLATE = "charuco"

# Generation of scene

## Basic scene

Load the basic scene into which other models will be included.

In [5]:
scene=mujoco.MjSpec.from_file(MODEL_PATH + "scene_high_resolution.xml")
scene.copy_during_attach = True

You can check the scene model file using `print_xml`.

In [6]:
print_xml(scene.to_xml())

## Robot

### Select robot

Load robot specification and inspect the body tree of the robot model using `print_body_tree`.

In [7]:
robot_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{ROBOT}.xml")
robot_spec.copy_during_attach = True
print_body_tree(robot_spec.worldbody, robot_spec)


Body Tree for "world"
-base
--link0
---link1 (Joints: joint1-HINGE[Actuator: pos_joint1])
----link2 (Joints: joint2-HINGE[Actuator: pos_joint2])
-----link3 (Joints: joint3-HINGE[Actuator: pos_joint3])
------link4 (Joints: joint4-HINGE[Actuator: pos_joint4])
-------link5 (Joints: joint5-HINGE[Actuator: pos_joint5])
--------link6 (Joints: joint6-HINGE[Actuator: pos_joint6])
---------link7 (Joints: joint7-HINGE[Actuator: pos_joint7])
----------flange


Prepare data for model keyframes

In [8]:
robot_ctrl = np.array(robot_spec.keys[0].ctrl)
robot_qpos = np.array(robot_spec.keys[0].qpos)

### Attach robot to ground

Define the frame in the world where the robot will be attached.

In [9]:
attachment_frame = scene.worldbody.add_frame(pos=[0, 0, 0], axisangle=[0, 0, 1, 0 * np.pi / 2])
attachment_frame.attach_body(robot_spec.body("base"), f"{ROBOT_NAME}_")

Rename base bode of the robot body tree (needed for automatic handle generation when creating robot object)

In [10]:
scene.body(f"{ROBOT_NAME}_base").name = ROBOT_NAME

If there is already in robot MJCF file a contact excluded which involves original robot base name, then it is necessary to rename body name in that definition

In [11]:
# scene.excludes[i].bodyname1 = ROBOT_NAME

Optionally exclude contacts between other objects. 

In [12]:
# scene.add_exclude(bodyname1=f"{ROBOT_NAME}", bodyname2="ground")

## Gripper

### Attach gripper to robot

Load gripper specification and prepare data for model keyframes

In [13]:
if GRIPPER is not None:
    gripper_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{GRIPPER}.xml")
    gripper_spec.copy_during_attach = True
    if len(gripper_spec.keys) > 0:
        gripper_ctrl = np.array(gripper_spec.keys[0].ctrl)
        gripper_qpos = np.array(gripper_spec.keys[0].qpos)
    else:
        gripper_ctrl = np.array([])
        gripper_qpos = np.array([])   


 Define attachment frame, attach gripper to robot flange, and redefine robot TCP. You can check the body tree of tool

In [14]:
if GRIPPER is not None:
    attachment_frame = scene.body(f"{ROBOT_NAME}_flange").add_frame(pos=[0, 0, 0], quat=[0.9238795, 0, 0, -0.3826834])
    attachment_frame.attach_body(gripper_spec.body("hand"), f'{ROBOT_NAME}_tool_')
    scene.delete(scene.site(f"{ROBOT_NAME}_TCP"))
    print_body_tree(scene.worldbody, scene)



Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_tool_hand
------------panda_tool_left_finger (Joints: panda_tool_finger_joint1-SLIDE)
------------panda_tool_right_finger (Joints: panda_tool_finger_joint2-SLIDE)


### Attach tool to gripper

Load tool. You can check the body tree of tool

In [15]:
if TOOL is not None:
    tool_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{TOOL}.xml")
    tool_spec.copy_during_attach = True
    gripper_tcp_id = scene.site(f"{ROBOT_NAME}_tool_hand_TCP").id
    print_body_tree(tool_spec.worldbody, tool_spec)


Body Tree for "world"
-calib_charuco
-calib_checker


Define the attachment frame where the tool will be attached to the gripper, and attach tool.

In [16]:
if TOOL is not None:
    scene.site(f"{ROBOT_NAME}_tool_hand_TCP").pos = scene.site(f"{ROBOT_NAME}_tool_hand_TCP").pos + [0, 0, tool_spec.geom(f"{PLATE}_pattern").size[1]]
    scene.site(f"{ROBOT_NAME}_tool_hand_TCP").quat = rot_y(np.pi / 2)    
    attachment_frame = scene.body(f"{ROBOT_NAME}_tool_hand").add_frame(pos=scene.site(f"{ROBOT_NAME}_tool_hand_TCP").pos, quat=scene.sites[gripper_tcp_id].quat)
    attachment_frame.attach_body(tool_spec.body(f"calib_{PLATE}"), "plate_")


Rename tool site to define robot TCP (for automatic detection of TCP)

In [17]:
scene.site(f"{ROBOT_NAME}_tool_hand_TCP").name = f"{ROBOT_NAME}_TCP"


Check the body tree

In [18]:

print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_tool_hand
------------panda_tool_left_finger (Joints: panda_tool_finger_joint1-SLIDE)
------------panda_tool_right_finger (Joints: panda_tool_finger_joint2-SLIDE)
------------plate_calib_charuco


## Cameras

Load camera specification

In [19]:
if CAMERA is not None:
    camera_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{CAMERA}.xml")
    camera_spec.copy_during_attach = True
    camera_spec.body("cam").mocap=0
    print_body_tree(camera_spec.worldbody, camera_spec)

Body Tree for "world"
-cam


Define attachment frame in the world and attach camera

In [20]:
if CAMERA is not None:
    attachment_frame = scene.worldbody.add_frame(pos=[1, 0, 0.6], axisangle=[0, 0, 1, 2 * np.pi / 2])
    attachment_frame.attach_body(camera_spec.body("cam"), "cam_")

## Prepare MJCF XML file

Redefine MJCF keys. Delete all keys and define Key 0 as the initial configuration

In [21]:
scene.keys

[]

In [22]:
_tmp = scene.to_xml()
while len(scene.keys)>1:
    scene.delete(scene.keys[-1])
for k in scene.keys:
    k.ctrl = np.concatenate([robot_ctrl, gripper_ctrl, np.zeros(len(scene.actuators) - len(robot_ctrl) - len(gripper_ctrl))])

Define the camera body as `mocap` so that it can be moved within the application

In [23]:
scene.body("cam_cam").mocap=1

Save MJCF scene to XML

In [24]:
scene.modelname = f"calibration_{PLATE}_scene"
scene.option.timestep = 0.001
with open(MODEL_PATH + f"{scene.modelname}.xml", "w") as f:
    f.write(scene.to_xml())

# Scene and robot definition verification

This section performs a minimal end-to-end check that the generated MJCF scene and the robot wrapper are consistent.

What is done:

- The generated scene file `<ROBOT>_scene.xml` is loaded into `mujoco_scene`.
- The corresponding Python robot object is created on top of this scene.
- `JointNames='auto'` is used so that the wrapper discovers the prefixed joint names directly from the generated scene.
- A simple motion command `JMove(r.q_home)` is executed to verify that joints, actuators, and robot handles are connected correctly.
- Finally, the current Cartesian pose `r.x` is printed to confirm that forward kinematics and TCP-related model definitions are valid.

In practice, this block verifies that scene generation added the expected prefixes correctly and that the resulting model can be used immediately through the robot interface.

In [ ]:
from robotblockset.mujoco.robots_pymujoco import *
scene = mujoco_scene(MODEL_PATH + f"{ROBOT}_scene.xml", show_camera=None)

In [ ]:
r=eval(f"{ROBOT}(scene=scene, JointNames='auto')")

[RBS_INFO] [1773476082.781757593] [panda_PyMuJoCo]: Robot connected to MuJoCo


In [ ]:
r.JMove(r.q_home)
print(r.x)

[ 0.3067 -0.0000  0.4897 -0.0002 -1.0000 -0.0035  0.0046]


In [ ]:
print(r.Kinmodel()[0])

[ 0.3067 -0.0000  0.4897 -0.0002 -1.0000 -0.0035  0.0046]
